In [15]:
import gc
import os
import time
import random
import pickle
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sys import getsizeof
import tensorflow as tf
print(tf.__version__)
from tensorflow.keras import metrics
from tensorflow.keras.optimizers import Adam
from tensorflow.keras import layers, callbacks
from tensorflow.keras.models import Sequential, Model
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from tensorflow.keras import layers, models
from sklearn.preprocessing import LabelEncoder

! pip install tensorflow

2.20.0


KeyboardInterrupt: 

In [ ]:
#Definicje i założenia
nicki_badanych = ["abc", "Bear", "fghx", "jt", "mi2", "miguel", "mole", "Reshi", "sapling"]

przedrostek_pliku_danych = "plikWynikowy"
przedrostek_pliku_etykiet = "plikWynikowyET"
przedrostek_pliku_stat = "paramerty_"
nazwa_pliku = "_Dane32Przetworzone"
oznaczenie = "AVG"
etapy_pliku = ["TRAIN" , "TEST"]

przebiegi = [[0,1],[2,3], [4,5],[6], [7,8]]
przebieg_testowy = len(przebiegi)-1

sciezka_wag = "dogotowywane.weights.h5"

In [ ]:
#KONFIGUROWWALNE PARAMETRYY
dane_wieloplikowe = True
nr_przebiegu = 3 #  0-dane 0-3    |   1-> dane 4-6  |  2 -> dane tesowe , przebieg testowy
etap_pliku = etapy_pliku[0]
# 0 trenuj i zapisz niedogotowany model
# 1 trenuj i zapisz dogotowany model
# 2 nie trenuj, tylko przetestuj i zrob wykresy

In [ ]:
#Łączenie z dyskiem
from google.colab import drive
drive.mount('/content/drive')

dataSourcePath = '/content/drive/MyDrive/BIAI/'

In [ ]:
X = []
y_txt = []

if dane_wieloplikowe:
  parametry_stat = np.load(dataSourcePath+przedrostek_pliku_stat+etap_pliku+".npy")
  srednia = parametry_stat[0]
  odch_std = parametry_stat[1]

  for nr_plikow in przebiegi[nr_przebiegu]:
    nx = np.load(dataSourcePath+przedrostek_pliku_danych +str(nr_plikow)+"_"+etap_pliku+nazwa_pliku+oznaczenie+".npy", mmap_mode='r')
    ey = np.load(dataSourcePath+przedrostek_pliku_etykiet+str(nr_plikow)+"_"+etap_pliku+nazwa_pliku+oznaczenie+".npy", mmap_mode='r')
    #nx = (nx-srednia)/odch_std
    X.append(nx)
    y_txt.append(ey)

    del nx
    del ey
    gc.collect()
  X = np.concatenate(X)
  y_txt = np.concatenate(y_txt)
  #X = np.transpose(X, (0,2,3,1))

else:
  X = np.load(dataSourcePath+"abc_Dane32PrzetworzoneAVGlog.npy")
  y_txt = np.load(dataSourcePath+"abc_EtykietyDanychlog.npy", allow_pickle=True)
  X = np.transpose(X, (0,2,3,1))

input_shape = X.shape[1:]   # Wymiary pojedynczego obrazu


In [ ]:
print(X.shape)
print(y_txt.shape)
print(input_shape)

In [ ]:
labelEnc = LabelEncoder()
y = labelEnc.fit_transform(y_txt)
l_klas = len(labelEnc.classes_)
print(labelEnc.classes_)

In [ ]:
if not dane_wieloplikowe:
  X_train_validate, X_test, y_train_validate, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
  X_train, X_validate, y_train, y_validate = train_test_split(X_train_validate, y_train_validate, test_size=0.25, random_state=42)

  #Normalizacja - standaryzacja (Z-score)
  srednia = np.mean(X_train, axis=(0,1,2), keepdims=True) # 0-epoka, 1-czestotliwosc, 2-czas <-Do usrednienia     3-kanal
  std = np.std(X_train, axis=(0,1,2), keepdims=True )

  print(srednia.shape)
  print("Srednie dla 21 kanalow: \n")
  print(srednia[0][0][0])

  X_train= (X_train - srednia)/std
  X_validate = (X_validate - srednia)/std
  X_test = (X_test - srednia)/std
else:
  if nr_przebiegu == przebieg_testowy:
    X_test = X
    y_test = y

In [ ]:

model = models.Sequential()
model.add(layers.Conv2D(32, (1, 10), activation=None, padding='same', input_shape=input_shape))
model.add(layers.BatchNormalization())#  #
model.add(layers.Activation('relu'))
model.add(layers.MaxPooling2D((2,2)))#   #
model.add(layers.Conv2D(64, (3,1), activation=None, padding='same'))
model.add(layers.BatchNormalization())#  #
model.add(layers.Activation('relu'))
model.add(layers.MaxPooling2D((2,3)))
model.add(layers.Dropout(0.2)) # (anti-overfitting)
model.add(layers.DepthwiseConv2D(64, (3,3), activation=None, depth_multiplier=2, padding='same'))
model.add(layers.BatchNormalization())#  #
model.add(layers.Activation('relu'))
model.add(layers.DepthwiseConv2D(64, (2,2), activation=None, padding='same'))
model.add(layers.BatchNormalization())#  #
model.add(layers.Activation('relu'))
model.add(layers.MaxPooling2D((2,2)))#   #

model.add(layers.Flatten())
  #model.add(layers.GlobalAveragePooling2D())
model.add(layers.Dense(128, activation=None))
  #model.add(layers.BatchNormalization())
model.add(layers.Activation('relu'))
model.add(layers.Dropout(0.3)) # (anti-overfitting)
model.add(layers.Dense(l_klas, activation='softmax'))

optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)

model.compile(optimizer = optimizer , loss = 'sparse_categorical_crossentropy' ,
              metrics = ['accuracy'],#, metrics.Precision(), metrics.Recall(), metrics.AUC()],
              jit_compile=False)
model.summary()

ReduceLROnPlateau_callback = callbacks.ReduceLROnPlateau(
      monitor='val_accuracy',
      patience = 5,
      verbose=1,
      factor=0.3,
      min_lr=0.0000001)

EarlyStopping_callback = callbacks.EarlyStopping(
      monitor='val_loss',
      patience=10,
      start_from_epoch = 15,
      restore_best_weights=True,
      verbose=0,
      mode='min')

if nr_przebiegu != 0 and dane_wieloplikowe:
    model.load_weights(dataSourcePath+sciezka_wag)
    print("Poprwanie wczytano wagi!")
else:
    print("Utworzono nowy model!")

In [10]:
epochs = 70
batch_size=32  #16 # im mniejsza, tym większa dokładność, ale i więcej czasu

if dane_wieloplikowe:
  print("Poczatek treningu")
  hist = model.fit(
      x=X,
      y=y,
      batch_size=batch_size,
      epochs=epochs,
      verbose=1,
      callbacks=[ReduceLROnPlateau_callback, EarlyStopping_callback],
      #callbacks=[ReduceLROnPlateau_callback],
      shuffle=True
  )
  model.save_weights(dataSourcePath+sciezka_wag)
  print("Trening zakonczony, model zapisany")
else:
  print("Poczatek treningu (bez walidacji)")
  hist = model.fit(
      x=X_train,
      y=y_train,
      batch_size=batch_size,
      epochs=epochs,
      verbose=1,
      callbacks=[ReduceLROnPlateau_callback, EarlyStopping_callback],
      #callbacks=[ReduceLROnPlateau_callback],
      validation_data=(X_validate, y_validate),
      shuffle=True
  )

Poczatek treningu
Epoch 1/70
127/127 ━━━━━━━━━━━━━━━━━━━━ 15s 49ms/step - accuracy: 0.1300 - loss: 2.4632 - learning_rate: 0.0010
Epoch 2/70
  2/127 ━━━━━━━━━━━━━━━━━━━━ 7s 60ms/step - accuracy: 0.1328 - loss: 2.2967  

/usr/local/lib/python3.13/dist-packages/keras/src/callbacks/callback_list.py:171: UserWarning: Learning rate reduction is conditioned on metric `val_accuracy` which is not available. Available metrics are: accuracy,loss,learning_rate.
  callback.on_epoch_end(epoch, logs)
/usr/local/lib/python3.13/dist-packages/keras/src/callbacks/early_stopping.py:99: UserWarning: Early stopping conditioned on metric `val_loss` which is not available. Available metrics are: accuracy,loss,learning_rate
  current = self.get_monitor_value(logs)


127/127 ━━━━━━━━━━━━━━━━━━━━ 7s 51ms/step - accuracy: 0.1481 - loss: 2.3429 - learning_rate: 0.0010
Epoch 3/70
127/127 ━━━━━━━━━━━━━━━━━━━━ 6s 48ms/step - accuracy: 0.1545 - loss: 2.3080 - learning_rate: 0.0010
Epoch 4/70
127/127 ━━━━━━━━━━━━━━━━━━━━ 7s 51ms/step - accuracy: 0.1610 - loss: 2.2802 - learning_rate: 0.0010
Epoch 5/70
127/127 ━━━━━━━━━━━━━━━━━━━━ 6s 48ms/step - accuracy: 0.1738 - loss: 2.2528 - learning_rate: 0.0010
Epoch 6/70
127/127 ━━━━━━━━━━━━━━━━━━━━ 7s 52ms/step - accuracy: 0.1800 - loss: 2.2316 - learning_rate: 0.0010
Epoch 7/70
127/127 ━━━━━━━━━━━━━━━━━━━━ 6s 48ms/step - accuracy: 0.1897 - loss: 2.2044 - learning_rate: 0.0010
Epoch 8/70
127/127 ━━━━━━━━━━━━━━━━━━━━ 10s 50ms/step - accuracy: 0.1974 - loss: 2.1906 - learning_rate: 0.0010
Epoch 9/70
127/127 ━━━━━━━━━━━━━━━━━━━━ 6s 51ms/step - accuracy: 0.1917 - loss: 2.1749 - learning_rate: 0.0010
Epoch 10/70
127/127 ━━━━━━━━━━━━━━━━━━━━ 6s 50ms/step - accuracy: 0.2053 - loss: 2.1509 - learning_rate: 0.0010
Epoch 11/7

In [11]:
if nr_przebiegu == przebieg_testowy or not dane_wieloplikowe:
  test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
  print(f"Skuteczność na danych testowych: {test_acc*100:.2f}%")


In [12]:
if nr_przebiegu == przebieg_testowy or not dane_wieloplikowe:
  y_predicted = model.predict(X_test)
  y_p_argmax = np.argmax(y_predicted, axis=1)
  #y_test_argmax = np.argmax(y_test.to_numpy(), axis=1)
  categories=["abstract","airplane","apple","banana","bird","boat","car","dog","person","train","zebra"]

  print(classification_report(y_test,y_p_argmax))
  matrix = confusion_matrix(y_test, y_p_argmax)

  disp = ConfusionMatrixDisplay(matrix, display_labels=categories)
  fig, ax = plt.subplots(figsize=(12,12))
  disp.plot(ax=ax, cmap="plasma") #viridis , plasma
  plt.xticks(rotation=45, ha='right')
  plt.show()
  plt.close()

In [13]:
if nr_przebiegu == przebieg_testowy or not dane_wieloplikowe:
  wyk, (os1,os2) = plt.subplots(1,2, figsize=(20,5))
  os1.plot(hist.history['loss'],label='Strata',color='#AA0000',linewidth=2)
  os1.plot(hist.history['val_loss'],label='StrataVal',color='#FFAA00',linewidth=2)
  os1.set_title('Strata')
  os1.set_xlabel('Epoka',fontsize=10)
  os1.set_ylabel('Wartosc',fontsize=10)
  os1.grid(True,linestyle='--',alpha=0.6)
  os1.legend(fontsize=10)

  os2.plot(hist.history['accuracy'],label='Celność',color='#00AA00',linewidth=2)
  os2.plot(hist.history['val_accuracy'],label='CelnośćVal',color='#AAFF00',linewidth=2)
  os2.set_title('Celnosc')
  os2.set_xlabel('Epoka',fontsize=10)
  os2.set_ylabel('Wartosc',fontsize=10)
  os2.grid(True,linestyle='--',alpha=0.6)
  os2.legend(fontsize=10)

In [14]:
#model.save(dataSourcePath + "Najlepszy3-09-0_44" + ".keras")
#zaladowany = keras.models.load_model(dataSourcePath+nazwa+".keras")